# Phase 10 — the causal test: ablate the takeover experts

Phases 9/9b established that at the language-switch token, experts unused in the
recent working set take over the leading routing slots. This notebook asks the
causal question: **if those takeover experts are masked during generation, does
the language switch stop happening?**

Design (three arms, so the answer is causal and specific):
* **sham** — no ablation: the baseline switch rate (thinking off, the ~12% regime).
* **targeted** — the recurring takeover experts (identified offline from the
  phase-9b captures) are banned from routing in every generation step.
* **random** — an equal-count random set of *other* experts is banned per layer:
  controls for "any ablation disrupts the model".

Verdict logic: switch rate `targeted ≪ sham` while `random ≈ sham` ⇒ the takeover
experts are causally necessary for the switch. If `random` also drops, the effect
is unspecific disruption. Two built-in safety gates: a **consistency gate**
(are takeover experts shared across transcripts at all?) and a **hook
verification** (the ban provably changes both the routed top-k *and* the model's
output logits before any generation is trusted).

Practical note that makes this A100-feasible: the switch fires at answer token ~1,
so `MAX_NEW_TOKENS=64` suffices to classify a completion — generation stays short.

Cell 2 is offline (needs only `routing_caps/`). Cells 3–7 need the A100.


In [ ]:
# === Cell 1 — config =========================================================
import os
try:
    from google.colab import userdata
    v=None
    try: v=userdata.get("HF_TOKEN")
    except Exception: v=None
    if v: os.environ.setdefault("HF_TOKEN",v)
except Exception as e:
    print("colab secrets unavailable:", e)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")

MODEL       = os.environ.get("WEIRDSPEC_TARGET_MODEL","Qwen/Qwen3.6-35B-A3B-FP8")
DATA_DIR    = "/content/drive/MyDrive/weirdspec"
CAPTURE_DIR = "/content/drive/MyDrive/weirdspec/routing_caps"      # phase-9b captures
BANNED_JSON = "/content/drive/MyDrive/weirdspec/banned_experts.json"
# v2: the v1 run used top_p=0.95 (+ the model's default top_k), which truncates
# exactly the sampling tail the switch lives in -> sham baseline collapsed to 1%.
OUTPUT      = "/content/drive/MyDrive/weirdspec/ablation_results_v2.jsonl"   # resumable

# --- takeover-set derivation (Cell 2) ---
MIN_FRAC    = 0.20    # expert must be a takeover expert in >= this fraction of tipped transcripts
RANKS       = None    # None = all 8 ranks' new experts; or e.g. 4 = leading ranks 1..4 only
NOVELTY_WINDOW = 16

# --- generation experiment (Cells 6-7) ---
# Adaptive two-stage design (the v2 run showed ~2% sham on arbitrary prompts -
# no power): Cell 6 ranks candidate prompts by the pattern's OpenRouter
# replication rate, Cell 6b calibrates them sham-only and advances the
# switchiest to the 3-arm experiment.
N_CANDIDATES   = 18    # candidate prompts entering calibration
CAL_SAMPLES    = 8     # sham-only calibration samples per candidate
TOP_PROMPTS    = 6     # switchiest prompts advance to the 3-arm experiment
CAL_OUTPUT     = "/content/drive/MyDrive/weirdspec/ablation_calibration_v2.jsonl"
SAMPLES        = 24    # completions per (prompt, arm) in the 3-arm stage
MAX_NEW_TOKENS = 64    # switch fires at token ~1; short generations suffice
BATCH          = 8
TEMPERATURE    = 1.0   # WeirdChat protocol, thinking off
FOREIGN_RUN    = 3
SEED           = 1234
MOUNT_DRIVE    = True
print(f"arms: sham/targeted/random | {N_CANDIDATES} candidates -> calibration -> {TOP_PROMPTS} prompts "
      f"x {SAMPLES} samples x 3 arms (+{N_CANDIDATES*CAL_SAMPLES} calibration gens)")


In [ ]:
# === Cell 2 — OFFLINE: derive the takeover set + consistency gate ===========
# Needs only routing_caps/ (phase 9b). Saves banned_experts.json for the GPU part.
import os, json, numpy as np
if MOUNT_DRIVE and not os.path.exists(CAPTURE_DIR):
    try:
        from google.colab import drive; drive.mount("/content/drive")
    except Exception as e: print("mount skipped:", e)
def read_jsonl(path):
    rows=[]
    with open(path,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows
index=read_jsonl(os.path.join(CAPTURE_DIR,"index.jsonl"))
seen=set(); index=[r for r in index if not (r["id"] in seen or seen.add(r["id"]))]
tipped=[r for r in index if r["cls"]=="tipped" and r.get("tip_ai") is not None and r["tip_ai"]>=1]
print(f"{len(tipped)} tipped transcripts with a located switch")

W=NOVELTY_WINDOW
per_transcript_sets=[]           # list of {(layer, expert)} per transcript
for r in tipped:
    z=np.load(os.path.join(CAPTURE_DIR,r["file"])); R,A=z["R"],z["A"]
    p=r["tip_ai"]; t=int(A[p])
    prev=[int(A[j]) for j in range(max(0,p-W),p)]
    s=set()
    kk = R.shape[2] if RANKS is None else min(RANKS,R.shape[2])
    for l in range(R.shape[0]):
        past=set(int(x) for x in R[l,prev,:].ravel())
        for e in R[l,t,:kk]:
            if int(e) not in past: s.add((l,int(e)))
    per_transcript_sets.append(s)

from collections import Counter
cnt=Counter()
for s in per_transcript_sets: cnt.update(s)
need=int(np.ceil(MIN_FRAC*len(per_transcript_sets)))
banned={}
for (l,e),c in cnt.items():
    if c>=need: banned.setdefault(l,[]).append(int(e))
banned={l:sorted(v) for l,v in sorted(banned.items())}
n_banned=sum(len(v) for v in banned.values())

# consistency: how shared are the takeover sets between transcripts?
rng=np.random.default_rng(0); jac=[]
idxs=np.arange(len(per_transcript_sets))
for _ in range(min(400,len(idxs)*(len(idxs)-1)//2)):
    i,j=rng.choice(idxs,2,replace=False)
    a,b=per_transcript_sets[i],per_transcript_sets[j]
    if a|b: jac.append(len(a&b)/len(a|b))
print(f"takeover (layer,expert) pairs total: {len(cnt)} | recurring in >= {need}/{len(per_transcript_sets)} "
      f"transcripts: {n_banned} across {len(banned)} layers")
print(f"pairwise Jaccard of takeover sets: median {np.median(jac):.3f}  "
      f"[{np.percentile(jac,10):.3f}, {np.percentile(jac,90):.3f}]")
print("per-layer banned counts:", {l:len(v) for l,v in banned.items()})
print("top-10 recurring:", cnt.most_common(10))
if n_banned < 5:
    print("\n!! CONSISTENCY GATE: almost no shared takeover experts — the takeover set is")
    print("   idiosyncratic per transcript. A shared-mask ablation cannot work; that is a")
    print("   finding in itself. Consider RANKS=4 or lower MIN_FRAC before burning GPU time.")
else:
    os.makedirs(os.path.dirname(BANNED_JSON), exist_ok=True)
    json.dump(banned, open(BANNED_JSON,"w"))
    print(f"\nsaved -> {BANNED_JSON}")


In [ ]:
# === Cell 3 — mount + deps (GPU part starts here) ===========================
import os, sys, subprocess
if MOUNT_DRIVE:
    try:
        from google.colab import drive; drive.mount("/content/drive")
    except Exception as e: print("drive mount skipped:", e)
subprocess.run([sys.executable,"-m","pip","install","-q","transformers==5.10.2","accelerate"], check=True)
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())


In [ ]:
# === Cell 4 — load model (proven path) ======================================
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained(MODEL)
cfg = AutoConfig.from_pretrained(MODEL)
tc = getattr(cfg,"text_config",cfg)
def cfgget(o,*names):
    for n in names:
        v=getattr(o,n,None)
        if v is not None: return v
NUM_EXPERTS=cfgget(tc,"num_experts","n_routed_experts","num_local_experts")
TOP_K      =cfgget(tc,"num_experts_per_tok","num_experts_per_token","top_k")
HIDDEN     =cfgget(tc,"hidden_size"); N_LAYERS=cfgget(tc,"num_hidden_layers")
print(f"MoE: num_experts={NUM_EXPERTS} top_k={TOP_K} hidden={HIDDEN} layers={N_LAYERS}")
kwargs={"attn_implementation":"sdpa","device_map":"auto"}
kwargs["dtype"]="auto" if getattr(cfg,"quantization_config",None) is not None else torch.bfloat16
# generation needs the LM head -> ForCausalLM first, concrete-class fallback second
model=None
for loader in (AutoModelForCausalLM, AutoModel):
    try:
        model=loader.from_pretrained(MODEL, **kwargs); break
    except ValueError:
        continue
if model is None:
    import transformers as tf
    model=getattr(tf,str(cfg.architectures[0])).from_pretrained(MODEL, **kwargs)
assert hasattr(model,"generate"), "loaded class cannot generate - check architecture"
model.eval(); DEV=next(model.parameters()).device
print("loaded:", type(model).__name__, "on", DEV)


In [ ]:
# === Cell 5 — gates + ablation hooks + HARD verification ====================
import re, json, numpy as np, torch
def _wshape(mod):
    w=getattr(mod,"weight",None)
    return tuple(w.shape) if (w is not None and hasattr(w,"shape") and w.dim()==2) else None
def is_gate(name, shp, num_experts, hidden):
    if shp not in [(num_experts,hidden),(hidden,num_experts)]: return False
    low=name.lower()
    if any(b in low for b in ("shared","attn","proj")): return False
    return ("gate" in low or "router" in low or "moe" in low)
gates=[]
for name,mod in model.named_modules():
    if is_gate(name,_wshape(mod),NUM_EXPERTS,HIDDEN):
        m=re.search(r"layers\.(\d+)\.",name)
        gates.append((int(m.group(1)) if m else -1,name,mod))
gates.sort(key=lambda g:g[0])
assert gates, "no gates found"
print(f"{len(gates)} gates")

# Active ban masks: {layer_id: bool tensor [NUM_EXPERTS]}; empty dict = sham.
_BAN={}
def set_ban(banned_dict):
    _BAN.clear()
    for l,ids in (banned_dict or {}).items():
        m=torch.zeros(NUM_EXPERTS,dtype=torch.bool); m[list(map(int,ids))]=True
        _BAN[int(l)]=m
def _mask_logits(logits,m):
    return logits.masked_fill(m.to(logits.device), torch.finfo(logits.dtype).min/2)
def _mk_ablate(layer_id):
    def hook(module,inputs,output):
        m=_BAN.get(layer_id)
        if m is None: return None
        if torch.is_tensor(output):
            return _mask_logits(output,m)
        out=list(output); logits=_mask_logits(out[0],m); rest=out[1:]
        # if the tuple carries pre-derived top-k indices/weights, recompute them
        for i,t in enumerate(rest):
            if torch.is_tensor(t) and t.shape[-1]!=logits.shape[-1]:
                k=t.shape[-1]
                vals,idx=torch.topk(logits,k,dim=-1)
                if torch.is_floating_point(t): rest[i]=torch.softmax(vals,dim=-1).to(t.dtype)
                else:                          rest[i]=idx.to(t.dtype)
        return (logits,*rest)
    return hook
_cap={}
def _mk_cap(pos):
    def hook(module,inputs,output):
        logits=output[0] if isinstance(output,(tuple,list)) else output
        _cap[pos]=torch.topk(logits,TOP_K,dim=-1).indices.detach().to("cpu")
    return hook
# ORDER MATTERS: ablation hook first (replaces output), capture second (sees masked).
for i,(lid,name,mod) in enumerate(gates):
    mod.register_forward_hook(_mk_ablate(lid))
    mod.register_forward_hook(_mk_cap(i))

# ---------- verification: the ban must change routing AND the LM output ------
probe=tokenizer("The weather today is", return_tensors="pt").to(DEV)
# the router emits token-flattened logits [tokens, E] -> normalize captures to [tokens, K]
def _flat(c): return c.reshape(-1, TOP_K)
with torch.no_grad():
    set_ban({}); out_sham=model(**probe).logits[:, -1, :].float().cpu()
    routed_sham={i:_flat(_cap[i]).clone() for i in range(len(gates))}
    # ban each layer's current top-1 expert at the last token (guaranteed in-use)
    test_ban={gates[i][0]: [int(routed_sham[i][-1,0])] for i in range(len(gates))}
    set_ban(test_ban); out_abl=model(**probe).logits[:, -1, :].float().cpu()
    for i in range(len(gates)):
        lid=gates[i][0]
        assert int(test_ban[lid][0]) not in _flat(_cap[i])[-1].tolist(), \
            f"layer {lid}: banned expert still routed - hook ineffective"
    delta=float((out_sham-out_abl).abs().max())
    assert delta > 1e-3, f"LM logits unchanged under ablation (max delta {delta:.2e}) - ban not reaching the block"
set_ban({})
print(f"verification PASSED: banned experts leave the top-k and LM logits move (max delta {delta:.3f})")


In [ ]:
# === Cell 5b — dump router weights once (GPU; enables offline geometry) =====
import numpy as np, os
ROUTER_W_NPZ = "/content/drive/MyDrive/weirdspec/router_weights.npz"
Wd={}
for lid,name,mod in gates:
    w=mod.weight.detach().float().cpu().numpy()
    if w.shape[0]!=NUM_EXPERTS: w=w.T          # ensure [E, hidden]
    Wd[str(lid)]=w.astype(np.float16)
np.savez_compressed(ROUTER_W_NPZ, **Wd)
print(f"saved {len(Wd)} router matrices -> {ROUTER_W_NPZ} "
      f"({os.path.getsize(ROUTER_W_NPZ)/1e6:.0f} MB)")


In [ ]:
# === Cell 6 — candidate prompts, RANKED by replication rate ============
import os, json, hashlib, glob
def read_jsonl(path):
    rows=[]
    with open(path,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows
def find_file(d,*names):
    for nm in names:
        p=os.path.join(d,nm)
        if os.path.isfile(p): return p
    for nm in names:
        h=sorted(glob.glob(os.path.join(d,"**",nm),recursive=True),key=len)
        if h: return h[0]
    return None
FOREIGN=[(0x0370,0x03FF),(0x0400,0x04FF),(0x0500,0x052F),(0x0530,0x058F),(0x0590,0x05FF),
         (0x0600,0x06FF),(0x0700,0x074F),(0x0900,0x097F),(0x0E00,0x0E7F),
         (0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
def is_foreign(ch):
    if not ch.isalpha(): return False
    o=ord(ch)
    return False if o<0x0250 else any(a<=o<=b for a,b in FOREIGN)
def foreign_onset(text, run=3):
    cnt=0
    for ch in text:
        if is_foreign(ch):
            cnt+=1
            if cnt>=run: return True
        elif ch.isalpha(): cnt=0
    return False

wp=find_file(DATA_DIR,"weird_transcripts.jsonl"); mp=find_file(DATA_DIR,"weird_meta.jsonl")
trans=read_jsonl(wp); meta=read_jsonl(mp)
cand={}
for tr,m in zip(trans,meta):
    if m.get("behavior_id")!="language-switching-english": continue
    conv=tr["conversations"]
    last_u=max(i for i,t in enumerate(conv) if t["role"]=="user")
    msgs=conv[:last_u+1]
    key=hashlib.sha1("||".join(t["role"]+":"+t["content"] for t in msgs).encode()).hexdigest()[:12]
    rate=float(m.get("openrouter_rate") or 0.0)
    if key not in cand or rate>cand[key][0]:
        cand[key]=(rate,msgs)
candidates=sorted(cand.items(), key=lambda kv:-kv[1][0])[:N_CANDIDATES]
# default selection if calibration (Cell 6b) is skipped: top by replication rate
prompt_items=[(k,v[1]) for k,v in candidates[:TOP_PROMPTS]]
print(f"{len(candidates)} candidates (of {len(cand)} distinct prompts), ranked by OpenRouter replication:")
for k,(rate,msgs) in candidates:
    print(f"  {k}  rate={rate:.2f}  {msgs[-1]['content'][:70]!r}")


In [ ]:
# === Cell 6b — sham-only CALIBRATION -> pick the switchiest prompts =====
# Cheap stage 1: which candidate prompts actually switch under LOCAL protocol
# sampling? Only those advance to the 3-arm experiment (stage 2). Resumable.
import os, json, torch
def render(msgs):
    try:
        return tokenizer.apply_chat_template(msgs, tokenize=False,
                add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
tokenizer.padding_side="left"
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
done={(r["prompt_key"],r["sample_idx"]) for r in (read_jsonl(CAL_OUTPUT) if os.path.exists(CAL_OUTPUT) else [])}
outf=open(CAL_OUTPUT,"a",encoding="utf-8")
set_ban({})
total=len(candidates)*CAL_SAMPLES; n_done=len(done)
for pk,(rate,msgs) in candidates:
    text=render(msgs)
    todo=[s for s in range(CAL_SAMPLES) if (pk,s) not in done]
    for b0 in range(0,len(todo),BATCH):
        batch=todo[b0:b0+BATCH]
        enc=tokenizer([text]*len(batch), return_tensors="pt", padding=True).to(DEV)
        torch.manual_seed(SEED+hash(("cal",pk,b0))%10**6)
        with torch.no_grad():
            gen=model.generate(**enc, do_sample=True, temperature=TEMPERATURE,
                               top_p=1.0, top_k=0, repetition_penalty=1.0,
                               max_new_tokens=MAX_NEW_TOKENS,
                               pad_token_id=tokenizer.pad_token_id)
        for j,s in enumerate(batch):
            new=gen[j, enc["input_ids"].shape[1]:]
            ans=tokenizer.decode(new, skip_special_tokens=True)
            outf.write(json.dumps(dict(prompt_key=pk,sample_idx=s,
                switched=foreign_onset(ans), answer_chars=len(ans)),ensure_ascii=False)+"\n")
        outf.flush(); n_done+=len(batch)
        print(f"  [cal] {n_done}/{total}")
outf.close()

rows=read_jsonl(CAL_OUTPUT)
per={}
for r in rows:
    k,n=per.get(r["prompt_key"],(0,0)); per[r["prompt_key"]]=(k+int(r["switched"]),n+1)
ranked=sorted(candidates, key=lambda kv:(-per.get(kv[0],(0,0))[0], -kv[1][0]))
print(f"\n{'prompt':14s} {'cal switches':>13s} {'or-rate':>8s}")
for pk,(rate,msgs) in ranked:
    k,n=per.get(pk,(0,0))
    print(f"{pk:14s} {k:6d}/{n:<6d} {rate:8.2f}")
prompt_items=[(pk,msgs) for pk,(rate,msgs) in ranked[:TOP_PROMPTS]]
tot_k=sum(per.get(pk,(0,0))[0] for pk,_ in ranked[:TOP_PROMPTS])
tot_n=sum(per.get(pk,(0,0))[1] for pk,_ in ranked[:TOP_PROMPTS])
print(f"\nselected {len(prompt_items)} prompts; pooled calibration switch rate "
      f"{100*tot_k/max(tot_n,1):.1f}% ({tot_k}/{tot_n})")
if tot_k < 5:
    print("!! even the best prompts barely switch under LOCAL bf16 generation. More samples")
    print("   will not fix a ~0% base rate - the honest next step is the FP8-native box")
    print("   (L40S/H100 SkyPilot) where the checkpoint runs as the dataset saw it.")


In [ ]:
# === Cell 7 — generate under the three arms (resumable) =====================
import os, json, torch, numpy as np
banned=json.load(open(BANNED_JSON))
banned={int(l):v for l,v in banned.items()}
n_banned=sum(len(v) for v in banned.values())
# random arm: equal count per layer, drawn from the complement, fixed seed
rng=np.random.default_rng(SEED)
rand_ban={}
for l,ids in banned.items():
    pool=[e for e in range(NUM_EXPERTS) if e not in set(ids)]
    rand_ban[l]=sorted(int(x) for x in rng.choice(pool,size=len(ids),replace=False))
ARMS={"sham":{}, "targeted":banned, "random":rand_ban}
print(f"targeted: {n_banned} experts over {len(banned)} layers | random arm matched per layer")

FOREIGN=[(0x0370,0x03FF),(0x0400,0x04FF),(0x0500,0x052F),(0x0530,0x058F),(0x0590,0x05FF),
         (0x0600,0x06FF),(0x0700,0x074F),(0x0900,0x097F),(0x0E00,0x0E7F),
         (0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
def is_foreign(ch):
    if not ch.isalpha(): return False
    o=ord(ch)
    return False if o<0x0250 else any(a<=o<=b for a,b in FOREIGN)
def foreign_onset(text, run=FOREIGN_RUN):
    cnt=0
    for ch in text:
        if is_foreign(ch):
            cnt+=1
            if cnt>=run: return True
        elif ch.isalpha(): cnt=0
    return False

def render(msgs):
    try:
        return tokenizer.apply_chat_template(msgs, tokenize=False,
                add_generation_prompt=True, enable_thinking=False)
    except TypeError:   # template without the kwarg
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

tokenizer.padding_side="left"
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
done={(r["arm"],r["prompt_key"],r["sample_idx"]) for r in (read_jsonl(OUTPUT) if os.path.exists(OUTPUT) else [])}
outf=open(OUTPUT,"a",encoding="utf-8")
total=len(ARMS)*len(prompt_items)*SAMPLES; n_done=len(done)
for arm,ban in ARMS.items():
    set_ban(ban)
    for pk,msgs in prompt_items:
        text=render(msgs)
        todo=[s for s in range(SAMPLES) if (arm,pk,s) not in done]
        for b0 in range(0,len(todo),BATCH):
            batch=todo[b0:b0+BATCH]
            enc=tokenizer([text]*len(batch), return_tensors="pt", padding=True).to(DEV)
            torch.manual_seed(SEED+hash((arm,pk,b0))%10**6)
            with torch.no_grad():
                # WeirdChat protocol: temperature 1.0, NO truncation. top_p/top_k/
                # repetition_penalty are set explicitly so the model's own
                # generation_config defaults (e.g. top_k=20) cannot sneak in and
                # cut the sampling tail - the switch IS a tail event.
                gen=model.generate(**enc, do_sample=True, temperature=TEMPERATURE,
                                   top_p=1.0, top_k=0, repetition_penalty=1.0,
                                   max_new_tokens=MAX_NEW_TOKENS,
                                   pad_token_id=tokenizer.pad_token_id)
            for j,s in enumerate(batch):
                new=gen[j, enc["input_ids"].shape[1]:]
                ans=tokenizer.decode(new, skip_special_tokens=True)
                outf.write(json.dumps(dict(arm=arm,prompt_key=pk,sample_idx=s,
                    switched=foreign_onset(ans), answer_chars=len(ans),
                    answer=ans[:400]),ensure_ascii=False)+"\n")
            outf.flush(); n_done+=len(batch)
            print(f"  [{arm}] {n_done}/{total}")
set_ban({})
outf.close(); print("results ->", OUTPUT)


In [ ]:
# === Cell 8 — analysis: is the switch causally carried by the takeover set? =
import os, json
import numpy as np, matplotlib.pyplot as plt
def read_jsonl(path):
    rows=[]
    with open(path,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows
OUTPUT = globals().get("OUTPUT","/content/drive/MyDrive/weirdspec/ablation_results.jsonl")
if not os.path.exists(OUTPUT):
    from google.colab import drive; drive.mount("/content/drive")
rows=read_jsonl(OUTPUT)
rows=[r for r in rows if r["answer_chars"]>=10]      # drop empty generations
def wilson(k,n,z=1.96):
    if n==0: return 0,0,0
    p=k/n; d=1+z*z/n
    c=(p+z*z/(2*n))/d; h=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return p,max(0,c-h),min(1,c+h)
def diff_ci(k1,n1,k2,n2,boot=4000,seed=0):
    rng=np.random.default_rng(seed)
    a=np.zeros(n1); a[:k1]=1; b=np.zeros(n2); b[:k2]=1
    d=[np.mean(rng.choice(a,n1))-np.mean(rng.choice(b,n2)) for _ in range(boot)]
    return k1/max(n1,1)-k2/max(n2,1), np.percentile(d,2.5), np.percentile(d,97.5)
stats={}
print(f"{'arm':10s} {'n':>5} {'switched':>9} {'rate':>7} {'95% CI':>18}")
for arm in ("sham","targeted","random"):
    g=[r for r in rows if r["arm"]==arm]
    k=sum(r["switched"] for r in g); n=len(g)
    p,lo,hi=wilson(k,n); stats[arm]=(k,n,p,lo,hi)
    print(f"{arm:10s} {n:5d} {k:9d} {100*p:6.1f}% [{100*lo:5.1f},{100*hi:5.1f}]")
if all(a in stats and stats[a][1]>0 for a in ("sham","targeted","random")):
    kt,nt=stats["targeted"][:2]; ks,ns=stats["sham"][:2]; kr,nr=stats["random"][:2]
    dt,tl,th=diff_ci(kt,nt,ks,ns); dr,rl,rh=diff_ci(kr,nr,ks,ns)
    dtr,trl,trh=diff_ci(kt,nt,kr,nr)          # targeted vs random, the SPECIFICITY test
    print(f"\ntargeted - sham   : {100*dt:+.1f}pp  95% CI [{100*tl:+.1f},{100*th:+.1f}]")
    print(f"random   - sham   : {100*dr:+.1f}pp  95% CI [{100*rl:+.1f},{100*rh:+.1f}]")
    print(f"targeted - random : {100*dtr:+.1f}pp  95% CI [{100*trl:+.1f},{100*trh:+.1f}]")
    # CAUSAL requires suppression vs sham AND vs the matched random ablation —
    # "random ~ sham looked non-significant" alone is absence of evidence, not specificity.
    if th<0 and trh<0:
        verdict="CAUSAL & SPECIFIC: the takeover experts suppress the switch, beyond matched random ablation."
    elif th<0 and rh<0:
        verdict="NON-SPECIFIC: both ablations suppress the switch - disruption, not the specific takeover set."
    elif th<0:
        verdict=("SUPPRESSION CONFIRMED, SPECIFICITY UNRESOLVED: targeted < sham is significant, but "
                 "targeted vs random is not - raise SAMPLES to power the specificity test.")
    elif tl<=0<=th:
        verdict="NO CAUSAL EFFECT DETECTED: the targeted ablation does not change the switch rate."
    else:
        verdict="UNEXPECTED: targeted ablation INCREASES switching - inspect samples."
    print("\nVERDICT:", verdict)
    fig,ax=plt.subplots(figsize=(6.2,4))
    arms=["sham","targeted","random"]
    ps=[100*stats[a][2] for a in arms]
    los=[100*(stats[a][2]-stats[a][3]) for a in arms]; his=[100*(stats[a][4]-stats[a][2]) for a in arms]
    ax.bar(range(3),ps,color=["#6B7280","#2563EB","#9CA3AF"],width=.6)
    ax.errorbar(range(3),ps,yerr=[los,his],fmt="none",ecolor="#111827",capsize=4)
    ax.set_xticks(range(3)); ax.set_xticklabels(arms)
    ax.set_ylabel("switch rate (%)"); ax.set_title("does banning the takeover experts stop the switch?")
    ax.grid(True,axis="y",alpha=.25)
    plt.tight_layout(); plt.show()
# text-quality sanity: ablation must not have destroyed generation
for arm in ("sham","targeted","random"):
    g=[r for r in rows if r["arm"]==arm]
    if g: print(f"sanity [{arm}]: mean answer length {np.mean([r['answer_chars'] for r in g]):.0f} chars")


## Relational metrics — the relationship between the active experts

Ablation asks "are these experts necessary?". The two cells below ask the
*relational* question: how do the active experts at the switch relate to each
other — an anomalous combination, or a pre-formed module?

* **Cell 9 (offline):** co-activation affinity `C[e,f]/min(n_e,n_f)` under two
  references — normal behaviour (controls) and the sustained foreign mode. The
  decisive pattern: switch coalition **anomalous under normal stats** but
  **already coherent under sustained-mode stats** = the mode arrives as a unit.
* **Cell 10 (offline, needs the Cell-5b weight dump):** gate-vector geometry —
  mean pairwise cosine within the coalition (directional coherence) and the
  cosine distance the routing centroid jumps at the switch.


In [ ]:
# === Cell 9 — RELATIONAL: co-activation coherence of the switch coalition ===
# Offline (needs only routing_caps/). Question: is the expert coalition that
# enters at the switch (a) an anomalous combination by NORMAL co-activation
# statistics, and (b) already the established clique of the SUSTAINED foreign
# mode - i.e. does the mode arrive as a pre-formed module, or as a jumble that
# only organises later?  Coherence(token) = mean pairwise co-activation affinity of its
# top-8 coalition, per layer, averaged over layers, under a reference
# co-activation statistic.  (Caveat: C_sustained is built from all tipped
# transcripts incl. the scored one - with ~30 transcripts the leave-one-out
# difference is negligible.)
import os, json, numpy as np
CAPTURE_DIR = globals().get("CAPTURE_DIR","/content/drive/MyDrive/weirdspec/routing_caps")
if not os.path.exists(CAPTURE_DIR):
    from google.colab import drive; drive.mount("/content/drive")
def _rj(path):
    rows=[]
    with open(path,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows
index=_rj(os.path.join(CAPTURE_DIR,"index.jsonl"))
_s=set(); index=[r for r in index if not (r["id"] in _s or _s.add(r["id"]))]
_c={}
def zl(rec):
    if rec["file"] not in _c:
        z=np.load(os.path.join(CAPTURE_DIR,rec["file"])); _c[rec["file"]]=(z["R"],z["A"])
        if len(_c)>300: _c.pop(next(iter(_c)))
    return _c[rec["file"]]
tipped=[r for r in index if r["cls"]=="tipped" and r.get("tip_ai") is not None and r["tip_ai"]>=1]
ctl=[r for r in index if r["cls"]=="control"]
L,E = zl(index[0])[0].shape[0], int(max(zl(index[0])[0].max()+1, 256))
SUB=2   # token subsampling for the reference statistics (speed)

def cooc(recs, tok_fn):
    """Per-layer co-occurrence counts C [L,E,E], marginals n [L,E], token count T."""
    C=np.zeros((L,E,E),np.float32); n=np.zeros((L,E),np.float32); T=0
    for rec in recs:
        R,A=zl(rec); idxs=tok_fn(rec)
        if not idxs: continue
        pos=np.array([int(A[i]) for i in idxs])
        T+=len(pos)
        for l in range(L):
            sel=R[l,pos,:].astype(np.int64)          # [t,8]
            Z=np.zeros((len(pos),E),np.float32)
            Z[np.arange(len(pos))[:,None],sel]=1.0
            C[l]+=Z.T@Z; n[l]+=Z.sum(0)
    return C,n,max(T,1)
def affinity(C,n,T):
    """Co-activation affinity in [0,1]: C[e,f] / min(n_e,n_f). Never-seen pairs
    (or experts) score 0 - unlike smoothed PMI, whose +1 floors give unseen
    pairs log(T), i.e. a MAXIMALLY positive score (verified failure mode)."""
    out=np.empty_like(C)
    for l in range(C.shape[0]):
        out[l]=C[l]/np.maximum(np.minimum.outer(n[l],n[l]),1.0)
    return out
def coherence(rec,i,P):
    R,A=zl(rec); t=int(A[i]); vals=[]
    for l in range(L):
        s=np.unique(R[l,t,:].astype(np.int64))
        if len(s)<2: continue
        M=P[l][np.ix_(s,s)]
        iu=np.triu_indices(len(s),1)
        vals.append(float(M[iu].mean()))
    return float(np.mean(vals)) if vals else None

print("building reference statistics (a few minutes)...")
C_ctl ,n_ctl ,T_ctl =cooc(ctl,    lambda r: list(range(1,r["n_assistant"],SUB)))
C_sus ,n_sus ,T_sus =cooc(tipped, lambda r: [i for i in r.get("foreign_ai",[]) if i>=r["tip_ai"]+20][::SUB])
P_ctl,P_sus=affinity(C_ctl,n_ctl,T_ctl),affinity(C_sus,n_sus,T_sus)
print(f"reference tokens: control={T_ctl}  sustained-foreign={T_sus}")

rng=np.random.default_rng(11)
tipsx=[r["tip_ai"] for r in tipped]
groups={}
groups["switch"]        =[(r,r["tip_ai"]) for r in tipped]
groups["matched-ctl"]   =[(c,int(p)) for c in ctl for p in [rng.choice(tipsx)] if c["n_assistant"]>p]
groups["sustained"]     =[(r,l[len(l)//2]) for r in tipped
                          for l in [[i for i in r.get("foreign_ai",[]) if i>=r["tip_ai"]+20]] if l]
def coh_list(pairs,P):
    out=[]
    for rec,i in pairs:
        v=coherence(rec,i,P)
        if v is not None: out.append(v)
    return np.array(out)
def dci(a,b,boot=3000,seed=3):
    rng=np.random.default_rng(seed)
    d=[np.mean(rng.choice(a,len(a)))-np.mean(rng.choice(b,len(b))) for _ in range(boot)]
    return float(np.mean(a)-np.mean(b)),float(np.percentile(d,2.5)),float(np.percentile(d,97.5))

print(f"\n{'coalition @':14s} {'ref: NORMAL':>14s} {'ref: SUSTAINED':>15s}   (mean pairwise co-activation affinity)")
res={}
for gname,pairs in groups.items():
    a=coh_list(pairs,P_ctl); b=coh_list(pairs,P_sus)
    res[gname]=(a,b)
    print(f"{gname:14s} {a.mean():14.3f} {b.mean():15.3f}   (n={len(a)})")
dA,lA,hA=dci(res["switch"][0],res["matched-ctl"][0])
dB,lB,hB=dci(res["switch"][1],res["matched-ctl"][1])
dC,lC,hC=dci(res["switch"][1],res["sustained"][1])
print(f"\n(A) switch vs matched-ctl under NORMAL stats   : {dA:+.3f} [{lA:+.3f},{hA:+.3f}]"
      + ("   sig" if hA<0 or lA>0 else ""))
print(f"(B) switch vs matched-ctl under SUSTAINED stats: {dB:+.3f} [{lB:+.3f},{hB:+.3f}]"
      + ("   sig" if hB<0 or lB>0 else ""))
print(f"(C) switch vs sustained   under SUSTAINED stats: {dC:+.3f} [{lC:+.3f},{hC:+.3f}]"
      + ("   sig" if hC<0 or lC>0 else ""))
if hA<0 and lB>0:
    print("\nREADING: the entering coalition is ANOMALOUS by normal co-activation standards but")
    print("ALREADY COHERENT by the sustained-mode's standards - the foreign mode arrives as a")
    print("pre-formed module at the switch token" + (", indistinguishable from its later self." if lC<=0<=hC else
          " (though still measurably different from its settled form)."))
elif hA<0:
    print("\nREADING: the entering coalition is anomalous under BOTH references - a jumbled entry;")
    print("the coherent mode only forms after the switch.")
else:
    print("\nREADING: no relational anomaly at entry - the individual-novelty story carries the effect.")


In [ ]:
# === Cell 10 — RELATIONAL: gate-vector geometry of the coalitions ===========
# Offline (needs router_weights.npz from Cell 5b + routing_caps/). Weight-based
# counterpart of Cell 9: cosine similarity between the gate row-vectors of the
# active experts. High mean pairwise cosine = the coalition listens to similar
# hidden-state directions (a directionally coherent module).  Also: cosine
# distance between the switch coalition's centroid and the pre-switch working
# set's centroid - the literal "distance the routing jumps" at the switch.
import os, json, numpy as np
ROUTER_W_NPZ = globals().get("ROUTER_W_NPZ","/content/drive/MyDrive/weirdspec/router_weights.npz")
CAPTURE_DIR  = globals().get("CAPTURE_DIR","/content/drive/MyDrive/weirdspec/routing_caps")
if not os.path.exists(ROUTER_W_NPZ):
    from google.colab import drive; drive.mount("/content/drive")
_wz=np.load(ROUTER_W_NPZ)
WN={int(k): (lambda w: w/np.maximum(np.linalg.norm(w,axis=1,keepdims=True),1e-8))(_wz[k].astype(np.float32))
    for k in _wz.files}
def _rj(path):
    rows=[]
    with open(path,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows
index=_rj(os.path.join(CAPTURE_DIR,"index.jsonl"))
_s=set(); index=[r for r in index if not (r["id"] in _s or _s.add(r["id"]))]
_c2={}
def zl(rec):
    if rec["file"] not in _c2:
        z=np.load(os.path.join(CAPTURE_DIR,rec["file"])); _c2[rec["file"]]=(z["R"],z["A"])
        if len(_c2)>300: _c2.pop(next(iter(_c2)))
    return _c2[rec["file"]]
tipped=[r for r in index if r["cls"]=="tipped" and r.get("tip_ai") is not None and r["tip_ai"]>=1]
ctl=[r for r in index if r["cls"]=="control"]
L=zl(index[0])[0].shape[0]; W=16

def pair_cos(rec,i):
    """Mean pairwise cosine within the top-8 coalition, averaged over layers."""
    R,A=zl(rec); t=int(A[i]); vals=[]
    for l in range(L):
        if l not in WN: continue
        V=WN[l][np.unique(R[l,t,:].astype(np.int64))]
        if V.shape[0]<2: continue
        S=V@V.T; iu=np.triu_indices(V.shape[0],1)
        vals.append(float(S[iu].mean()))
    return float(np.mean(vals)) if vals else None
def centroid_jump(rec,i):
    """Cosine distance between the coalition centroid at i and the centroid of
    the preceding window's working set (per layer, averaged)."""
    R,A=zl(rec)
    if i<1: return None
    prev=[int(A[j]) for j in range(max(0,i-W),i)]; t=int(A[i]); vals=[]
    for l in range(L):
        if l not in WN: continue
        cur=WN[l][np.unique(R[l,t,:].astype(np.int64))].mean(0)
        past=WN[l][np.unique(R[l,prev,:].ravel().astype(np.int64))].mean(0)
        cs=float(np.dot(cur,past)/max(np.linalg.norm(cur)*np.linalg.norm(past),1e-8))
        vals.append(1.0-cs)
    return float(np.mean(vals)) if vals else None

rng=np.random.default_rng(12); tipsx=[r["tip_ai"] for r in tipped]
groups={"switch":[(r,r["tip_ai"]) for r in tipped],
        "matched-ctl":[(c,int(p)) for c in ctl for p in [rng.choice(tipsx)] if c["n_assistant"]>p],
        "sustained":[(r,l[len(l)//2]) for r in tipped
                     for l in [[i for i in r.get("foreign_ai",[]) if i>=r["tip_ai"]+20]] if l]}
def collect(fn,pairs):
    out=[fn(rec,i) for rec,i in pairs]
    return np.array([v for v in out if v is not None])
def dci(a,b,boot=3000,seed=4):
    rng=np.random.default_rng(seed)
    d=[np.mean(rng.choice(a,len(a)))-np.mean(rng.choice(b,len(b))) for _ in range(boot)]
    return float(np.mean(a)-np.mean(b)),float(np.percentile(d,2.5)),float(np.percentile(d,97.5))

print(f"{'coalition @':14s} {'pairwise cos':>13s} {'centroid jump':>14s}")
res={}
for g,pairs in groups.items():
    pc=collect(pair_cos,pairs); cj=collect(centroid_jump,pairs)
    res[g]=(pc,cj)
    print(f"{g:14s} {pc.mean():13.4f} {cj.mean():14.4f}   (n={len(pc)})")
d1,l1,h1=dci(res["switch"][0],res["matched-ctl"][0])
d2,l2,h2=dci(res["switch"][1],res["matched-ctl"][1])
print(f"\npairwise-cos  switch - matched-ctl: {d1:+.4f} [{l1:+.4f},{h1:+.4f}]" + ("   sig" if h1<0 or l1>0 else ""))
print(f"centroid-jump switch - matched-ctl: {d2:+.4f} [{l2:+.4f},{h2:+.4f}]" + ("   sig" if h2<0 or l2>0 else ""))
print("\nreading: pairwise-cos HIGHER at the switch = the entering experts listen to similar")
print("directions (a directional module); centroid-jump HIGHER = the routing genuinely jumps")
print("a long way in gate space at the switch - the geometric size of the reroute.")


In [ ]:
# === Cell 11 — PORTRAIT of the takeover experts (offline) ===================
# Who are the ~317? (1) how often the model calls them: normally, BEFORE the
# switch (the critical-length window), at the switch, inside the mode;
# (2) dispersion vs polarisation of the set (egalitarian or core-periphery);
# (3) one module or per-language submodules (script-resolved overlap);
# (4) rank behaviour when they fire. Needs only Drive (captures + banned json
# + weird_transcripts for script labels) - runs on any runtime, no GPU.
import os, json, glob, warnings, numpy as np, matplotlib.pyplot as plt
warnings.filterwarnings("ignore", message=".*empty slice.*")
warnings.filterwarnings("ignore", message=".*Degrees of freedom.*")
CAPTURE_DIR=globals().get("CAPTURE_DIR","/content/drive/MyDrive/weirdspec/routing_caps")
BANNED_JSON=globals().get("BANNED_JSON","/content/drive/MyDrive/weirdspec/banned_experts.json")
DATA_DIR   =globals().get("DATA_DIR","/content/drive/MyDrive/weirdspec")
if not os.path.exists(CAPTURE_DIR):
    from google.colab import drive; drive.mount("/content/drive")
def _rj(path):
    rows=[]
    with open(path,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows
def _ff(d,*names):
    for nm in names:
        p=os.path.join(d,nm)
        if os.path.isfile(p): return p
    for nm in names:
        h=sorted(glob.glob(os.path.join(d,"**",nm),recursive=True),key=len)
        if h: return h[0]
    return None
index=_rj(os.path.join(CAPTURE_DIR,"index.jsonl"))
_s=set(); index=[r for r in index if not (r["id"] in _s or _s.add(r["id"]))]
_cz={}
def zl(rec):
    if rec["file"] not in _cz:
        z=np.load(os.path.join(CAPTURE_DIR,rec["file"])); _cz[rec["file"]]=(z["R"],z["A"])
        if len(_cz)>300: _cz.pop(next(iter(_cz)))
    return _cz[rec["file"]]
tipped=[r for r in index if r["cls"]=="tipped" and r.get("tip_ai") is not None and r["tip_ai"]>=1]
ctl=[r for r in index if r["cls"]=="control"]
R0,_=zl(index[0]); L,K=R0.shape[0],R0.shape[2]
E=max(256,int(R0.max())+1)
banned={int(l):sorted(v) for l,v in json.load(open(BANNED_JSON)).items()}
MASK=np.zeros((L,E),bool)
for l,ids in banned.items(): MASK[l,ids]=True
NB=int(MASK.sum()); BL=sorted(banned.keys())
print(f"{NB} takeover experts over {len(BL)} layers | layer sizes: "
      f"{ {l:len(v) for l,v in list(banned.items())[:8]} }...")

def occ(rec,i):
    """Fraction of top-8 slots (banned layers) held by takeover experts at token i."""
    R,A=zl(rec)
    if i<0 or i>=len(A): return None
    sel=R[np.array(BL),int(A[i]),:].astype(np.int64)
    return float(MASK[np.array(BL)[:,None],sel].mean())
def boot(vals,n=3000,seed=0):
    v=np.array([x for x in vals if x is not None])
    if len(v)<5: return None
    rng=np.random.default_rng(seed)
    bs=[np.mean(rng.choice(v,len(v))) for _ in range(n)]
    return float(v.mean()),float(np.percentile(bs,2.5)),float(np.percentile(bs,97.5)),len(v)

# ---- (1) usage by regime + the peri-switch curve ---------------------------
rng=np.random.default_rng(21)
tipsx=[r["tip_ai"] for r in tipped]
regs={}
regs["control (normal)"]=[occ(c,int(rng.integers(1,c["n_assistant"]))) for c in ctl]
regs["pre-switch (critical window)"]=[np.mean([occ(r,i) for i in range(max(0,r["tip_ai"]-10),r["tip_ai"])])
                                       if r["tip_ai"]>=1 else None for r in tipped]
regs["at the switch"]=[occ(r,r["tip_ai"]) for r in tipped]
regs["sustained (in the mode)"]=[occ(r,l[len(l)//2]) for r in tipped
     for l in [[i for i in r.get("foreign_ai",[]) if i>=r["tip_ai"]+20]] if l]
print(f"\n{'regime':32s} {'occupancy':>10s} {'95% CI':>18s} {'n':>4s}")
for k,v in regs.items():
    b=boot(v)
    if b: print(f"{k:32s} {100*b[0]:9.1f}% [{100*b[1]:5.1f},{100*b[2]:5.1f}] {b[3]:4d}")
DTS=list(range(-20,21))
M=np.full((len(tipped),len(DTS)),np.nan)
for a,r in enumerate(tipped):
    for di,dt in enumerate(DTS):
        v=occ(r,r["tip_ai"]+dt)
        if v is not None and r["tip_ai"]+dt>=0: M[a,di]=v
MC=np.full((len(ctl),len(DTS)),np.nan)
for a,c in enumerate(ctl):
    p=int(rng.choice(tipsx))
    if c["n_assistant"]<=p+20: continue
    for di,dt in enumerate(DTS):
        v=occ(c,p+dt)
        if v is not None and p+dt>=0: MC[a,di]=v
pre=boot([np.nanmean(row[:DTS.index(0)]) for row in M if not np.isnan(row[:DTS.index(0)]).all()],seed=1)
prec=boot([np.nanmean(row[:DTS.index(0)]) for row in MC if not np.isnan(row[:DTS.index(0)]).all()],seed=2)
if pre and prec:
    d=pre[0]-prec[0]
    print(f"\nPRECURSOR TEST (pre-switch minus control occupancy): {100*d:+.1f}pp")
    print("  -> ramp (early leak of the module) if clearly >0, step (silent until the switch) if ~0")

# ---- (2) polarisation: usage + takeover participation ----------------------
cnt_use=np.zeros((L,E)); T=0
for c in ctl:
    R,A=zl(c); pos=A[1::3]
    for l in BL:
        sel=R[l,pos.astype(int),:].astype(np.int64)
        np.add.at(cnt_use[l],sel.ravel(),1)
    T+=len(pos)
use_rate=np.array([cnt_use[l,e]/max(T*K,1) for l in BL for e in banned[l]])
W=16
part=np.zeros((L,E))
for r in tipped:
    R,A=zl(r); p=r["tip_ai"]
    prev=[int(A[j]) for j in range(max(0,p-W),p)]
    for l in BL:
        past=set(int(x) for x in R[l,prev,:].ravel())
        for e in R[l,int(A[p]),:]:
            if int(e) in banned.get(l,[]) and int(e) not in past: part[l,int(e)]+=1
part_counts=np.array([part[l,e] for l in BL for e in banned[l]])
def gini(x):
    x=np.sort(np.asarray(x,float)); n=len(x); s=x.sum()
    if s<=0: return 0.0
    return float((2*np.sum((np.arange(1,n+1))*x))/(n*s)-(n+1)/n)
print(f"\nPOLARISATION of the {NB}:")
print(f"  normal-usage rate: median {100*np.median(use_rate):.2f}% of control slots, "
      f"Gini {gini(use_rate):.2f}")
print(f"  takeover participation (of {len(tipped)} switches): median {np.median(part_counts):.0f}, "
      f"max {part_counts.max():.0f}, Gini {gini(part_counts):.2f}")
core=[(l,e,int(part[l,e])) for l in BL for e in banned[l] if part[l,e]>=0.5*len(tipped)]
print(f"  CORE (fires in >=50% of switches): {len(core)} experts: "
      f"{sorted(core,key=lambda x:-x[2])[:8]}")

# ---- (3) one module or per-language submodules? ----------------------------
wp=_ff(DATA_DIR,"weird_transcripts.jsonl")
SCRIPTS={"cyrillic":(0x0400,0x052F),"cjk":(0x3400,0x9FFF),"kana":(0x3040,0x30FF),
         "hangul":(0xAC00,0xD7AF),"greek":(0x0370,0x03FF),"arabic":(0x0600,0x06FF)}
def dominant_script(text):
    cnt={s:0 for s in SCRIPTS}
    for ch in text:
        o=ord(ch)
        for s,(a,b) in SCRIPTS.items():
            if a<=o<=b: cnt[s]+=1; break
    s,best=max(cnt.items(),key=lambda kv:kv[1])
    return s if best>=3 else None
script_of={}
if wp:
    for tr in _rj(wp):
        txt=" ".join(t["content"] for t in tr["conversations"] if t["role"]=="assistant")
        sc=dominant_script(txt)
        if sc: script_of[tr["id"]]=sc
tsets={}; tscript={}
for r in tipped:
    R,A=zl(r); p=r["tip_ai"]
    prev=[int(A[j]) for j in range(max(0,p-W),p)]
    s=set()
    for l in BL:
        past=set(int(x) for x in R[l,prev,:].ravel())
        s|={(l,int(e)) for e in R[l,int(A[p]),:] if int(e) not in past}
    tsets[r["id"]]=s
    if r["id"] in script_of: tscript[r["id"]]=script_of[r["id"]]
from collections import Counter
sc_counts=Counter(tscript.values())
print(f"\nSCRIPT LABELS: {dict(sc_counts)} (of {len(tipped)} tipped)")
wj,cj=[],[]
ids=[i for i in tsets if i in tscript]
for a in range(len(ids)):
    for b in range(a+1,len(ids)):
        A_,B_=tsets[ids[a]],tsets[ids[b]]
        if not (A_|B_): continue
        j=len(A_&B_)/len(A_|B_)
        (wj if tscript[ids[a]]==tscript[ids[b]] else cj).append(j)
if wj and cj:
    bw=boot(wj,seed=5); bc=boot(cj,seed=6)
    print(f"takeover-set Jaccard  WITHIN script: {bw[0]:.3f} [{bw[1]:.3f},{bw[2]:.3f}] (n={len(wj)})")
    print(f"                      CROSS  script: {bc[0]:.3f} [{bc[1]:.3f},{bc[2]:.3f}] (n={len(cj)})")
    print("  -> within >> cross  =  the 'module' decomposes into per-language submodules")
    print("     within ~= cross  =  one shared switching module across target languages")

# ---- (4) rank behaviour when they fire -------------------------------------
def rank_prof(pairs):
    P=np.zeros(K); n=0
    for rec,i in pairs:
        R,A=zl(rec)
        if i<0 or i>=len(A): continue
        sel=R[np.array(BL),int(A[i]),:].astype(np.int64)
        P+=MASK[np.array(BL)[:,None],sel].mean(axis=0); n+=1
    return P/max(n,1)
rp_switch=rank_prof([(r,r["tip_ai"]) for r in tipped])
rp_sust  =rank_prof([(r,l[len(l)//2]) for r in tipped
          for l in [[i for i in r.get("foreign_ai",[]) if i>=r["tip_ai"]+20]] if l])
rp_ctl   =rank_prof([(c,int(rng.integers(1,c["n_assistant"]))) for c in ctl])

fig,ax=plt.subplots(1,3,figsize=(15,4))
mu=np.nanmean(M,0); n_=np.maximum((~np.isnan(M)).sum(0),1); se=np.nanstd(M,0)/np.sqrt(n_)
ax[0].fill_between(DTS,mu-1.96*se,mu+1.96*se,color="#2563EB",alpha=.2,linewidth=0)
ax[0].plot(DTS,mu,color="#2563EB",lw=2,label="tipped")
muc=np.nanmean(MC,0); nc=np.maximum((~np.isnan(MC)).sum(0),1); sec=np.nanstd(MC,0)/np.sqrt(nc)
ax[0].fill_between(DTS,muc-1.96*sec,muc+1.96*sec,color="#6B7280",alpha=.15,linewidth=0)
ax[0].plot(DTS,muc,color="#6B7280",lw=2,ls="--",label="control (pseudo-tip)")
ax[0].axvline(0,color="black",lw=.8,ls=":")
ax[0].set_xlabel("token relative to the switch"); ax[0].set_ylabel("takeover-set occupancy")
ax[0].set_title("the critical window: silent, then step?"); ax[0].legend(frameon=False,fontsize=9)
ax[1].hist(part_counts,bins=np.arange(0,len(tipped)+2)-.5,color="#2563EB")
ax[1].set_xlabel(f"participates in N of {len(tipped)} switches"); ax[1].set_ylabel("experts")
ax[1].set_title(f"polarisation: Gini {gini(part_counts):.2f}")
x=np.arange(K); w=0.27
ax[2].bar(x-w,rp_switch,w,label="at switch",color="#2563EB")
ax[2].bar(x,   rp_sust,  w,label="sustained",color="#10B981")
ax[2].bar(x+w, rp_ctl,   w,label="control",color="#6B7280")
ax[2].set_xticks(x); ax[2].set_xticklabels([f"r{k+1}" for k in range(K)])
ax[2].set_ylabel("P(slot is takeover expert)"); ax[2].set_title("rank behaviour when they fire")
ax[2].legend(frameon=False,fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
# === Cell 12 — TOKENIZATION AUDIT: is the critical window an artifact? ======
# The model token is the mechanistic unit, but CJK/kana/hangul token fertility
# differs wildly from Latin, and with switches at assistant position ~1 a
# nominal 16-TOKEN window often holds 1-3 tokens. This cell (a) audits
# fertility per regime, (b) re-expresses the occupancy curve on tokenizer-
# independent axes (graphemes, UTF-8 bytes), (c) adds the grapheme-weighted
# mean, (d) sweeps the takeover-set definition over token AND content windows,
# and (e) sharpens temporal alignment: PREDECESSOR token (whose hidden state
# predicts the switch token) vs the token containing the first foreign letter
# vs the first fully-foreign token. Trigger evidence = the jump already at the
# predecessor; jump only at the foreign token = possibly a reaction to the
# script. Offline; needs captures + transcripts + the Qwen tokenizer (CPU ok).
import os, json, glob, warnings, numpy as np, matplotlib.pyplot as plt
warnings.filterwarnings("ignore", message=".*empty slice.*")
warnings.filterwarnings("ignore", message=".*Degrees of freedom.*")
try:
    import regex as _rx
    def graphemes(s): return _rx.findall(r"\X", s)
except Exception:
    def graphemes(s): return list(s)   # codepoint fallback (fine for CJK/kana/hangul)
CAPTURE_DIR=globals().get("CAPTURE_DIR","/content/drive/MyDrive/weirdspec/routing_caps")
BANNED_JSON=globals().get("BANNED_JSON","/content/drive/MyDrive/weirdspec/banned_experts.json")
DATA_DIR   =globals().get("DATA_DIR","/content/drive/MyDrive/weirdspec")
AUDIT_MAX_LEN=2048          # must equal the capture pass MAX_LEN
if not os.path.exists(CAPTURE_DIR):
    from google.colab import drive; drive.mount("/content/drive")
def _rj(path):
    rows=[]
    with open(path,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows
def _ff(d,*names):
    for nm in names:
        p=os.path.join(d,nm)
        if os.path.isfile(p): return p
    for nm in names:
        h=sorted(glob.glob(os.path.join(d,"**",nm),recursive=True),key=len)
        if h: return h[0]
    return None
index=_rj(os.path.join(CAPTURE_DIR,"index.jsonl"))
_s=set(); index=[r for r in index if not (r["id"] in _s or _s.add(r["id"]))]
_cz={}
def zl(rec):
    if rec["file"] not in _cz:
        z=np.load(os.path.join(CAPTURE_DIR,rec["file"])); _cz[rec["file"]]=(z["R"],z["A"])
        if len(_cz)>300: _cz.pop(next(iter(_cz)))
    return _cz[rec["file"]]
tipped=[r for r in index if r["cls"]=="tipped" and r.get("tip_ai") is not None and r["tip_ai"]>=1]
ctl=[r for r in index if r["cls"]=="control"]
R0,_=zl(index[0]); L,K=R0.shape[0],R0.shape[2]
E=max(256,int(R0.max())+1)
banned={int(l):sorted(v) for l,v in json.load(open(BANNED_JSON)).items()}
MASK=np.zeros((L,E),bool)
for l,ids in banned.items(): MASK[l,ids]=True
BL=np.array(sorted(banned.keys()))
CANON={(l,int(e)) for l in banned for e in banned[l]}
def occ(rec,i):
    R,A=zl(rec)
    if i<0 or i>=len(A): return None
    sel=R[BL,int(A[i]),:].astype(np.int64)
    return float(MASK[BL[:,None],sel].mean())
def boot_d(vals,n=3000,seed=0):
    v=np.array([x for x in vals if x is not None])
    if len(v)<5: return None
    rng=np.random.default_rng(seed)
    bs=[np.mean(rng.choice(v,len(v))) for _ in range(n)]
    return float(v.mean()),float(np.percentile(bs,2.5)),float(np.percentile(bs,97.5)),len(v)

if "tokenizer" not in globals():
    from transformers import AutoTokenizer
    tokenizer=AutoTokenizer.from_pretrained(globals().get("MODEL","Qwen/Qwen3.6-35B-A3B-FP8"))
FOR=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x06FF),
     (0x0700,0x074F),(0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),
     (0xAC00,0xD7AF),(0xF900,0xFAFF)]
def is_for(ch):
    if not ch.isalpha(): return False
    o=ord(ch)
    return False if o<0x0250 else any(a<=o<=b for a,b in FOR)

wp=_ff(DATA_DIR,"weird_transcripts.jsonl")
conv_of={tr["id"]:tr["conversations"] for tr in _rj(wp)} if wp else {}
def rebuild(rec):
    """Re-tokenize exactly as the capture pass did; return per-assistant-token
    char spans + strings, the onset char (assistant-relative) and integrity."""
    conv=conv_of.get(rec["id"])
    if conv is None: return None
    full="".join(f"<|im_start|>{t['role']}\n{t['content']}<|im_end|>\n" for t in conv)
    hdr="<|im_start|>assistant\n"; apos=full.rfind(hdr)
    a_char=apos+len(hdr) if apos>=0 else len(full)
    enc=tokenizer(full,return_offsets_mapping=True,truncation=True,max_length=AUDIT_MAX_LEN)
    offs=enc["offset_mapping"]
    A=[i for i,(s,e) in enumerate(offs) if s>=a_char and e>s]
    if len(A)!=rec["n_assistant"]: return None          # integrity guard
    spans=[(offs[i][0],offs[i][1],full[offs[i][0]:offs[i][1]]) for i in A]
    onset=None; cnt=0; start=None
    for i,ch in enumerate(full[a_char:]):
        if is_for(ch):
            if cnt==0: start=i
            cnt+=1
            if cnt>=3: onset=a_char+start; break
        elif ch.isalpha(): cnt=0; start=None
    return dict(spans=spans,onset_char=onset)

audit={}; n_fail=0
for rec in tipped+ctl:
    rb=rebuild(rec)
    if rb is None: n_fail+=1
    else: audit[rec["id"]]=rb
print(f"tokenization rebuilt for {len(audit)} transcripts ({n_fail} integrity failures skipped)")
def G(rec_id,i): return len(graphemes(audit[rec_id]["spans"][i][2]))
def B(rec_id,i): return len(audit[rec_id]["spans"][i][2].encode("utf-8"))

# ---- (a) fertility per regime + mid-token onset ----------------------------
def fert(tok_lists):
    g=[]; b=[]; n=0
    for rid,idxs in tok_lists:
        for i in idxs:
            g.append(G(rid,i)); b.append(B(rid,i)); n+=1
    g=np.array(g); b=np.array(b)
    return (100.0*n/max(g.sum(),1), float(np.median(g)), float(b.mean()), n)
T_aud=[r for r in tipped if r["id"] in audit]
C_aud=[c for c in ctl if c["id"] in audit]
pre_l =[(r["id"],list(range(max(0,r["tip_ai"]-10),r["tip_ai"]))) for r in T_aud]
sw_l  =[(r["id"],[r["tip_ai"]]) for r in T_aud]
sus_l =[(r["id"],[i for i in r.get("foreign_ai",[]) if i>=r["tip_ai"]+20][:60]) for r in T_aud]
ctl_l =[(c["id"],list(range(1,min(c["n_assistant"],80),3))) for c in C_aud]
print(f"\n{'regime':14s} {'tok/100 graphemes':>18s} {'med g/tok':>10s} {'bytes/tok':>10s} {'n_tok':>7s}")
for name,tl in (("control",ctl_l),("pre-switch",pre_l),("switch",sw_l),("sustained",sus_l)):
    t100,mg,bt,n=fert(tl)
    print(f"{name:14s} {t100:18.1f} {mg:10.1f} {bt:10.1f} {n:7d}")
mid=[]
for r in T_aud:
    a=audit[r["id"]]
    if a["onset_char"] is None: continue
    s,e,_=a["spans"][r["tip_ai"]]
    mid.append(s<a["onset_char"]<e)
print(f"mid-token onsets: {100*np.mean(mid):.0f}% of switches start MID-token "
      f"(first foreign letter not at a token boundary)")
eff=[min(r["tip_ai"],16) for r in T_aud]
print(f"effective pre-switch window: median {int(np.median(eff))} tokens (nominal 16) — "
      f"{100*np.mean(np.array(eff)<4):.0f}% of switches have <4 tokens of history")

# ---- (e) temporal alignment: predecessor / onset-token / fully-foreign -----
def ctl_occ_at(p,seed=7):
    vals=[occ(c,p) for c in C_aud if c["n_assistant"]>p]
    return float(np.mean([v for v in vals if v is not None])) if len(vals)>=10 else None
def fully_foreign_idx(r):
    a=audit[r["id"]]
    for i in range(r["tip_ai"],r["n_assistant"]):
        txt=a["spans"][i][2]; al=[ch for ch in txt if ch.isalpha()]
        if al and all(is_for(ch) for ch in al): return i
    return None
rows_t={"predecessor (t-1, predicts the switch)":lambda r: r["tip_ai"]-1,
        "token with first foreign letter (t)":  lambda r: r["tip_ai"],
        "first FULLY-foreign token":            fully_foreign_idx}
print(f"\n{'alignment':40s} {'occ':>7s} {'vs ctl@pos':>11s} {'95% CI':>18s} {'n':>4s}")
for name,fn in rows_t.items():
    diffs=[]
    for r in T_aud:
        i=fn(r)
        if i is None or i<0: continue
        o=occ(r,i); cm=ctl_occ_at(i)
        if o is not None and cm is not None: diffs.append((o,o-cm))
    if diffs:
        b=boot_d([d for _,d in diffs],seed=11)
        print(f"{name:40s} {100*np.mean([o for o,_ in diffs]):6.1f}% {100*b[0]:+10.1f}pp "
              f"[{100*b[1]:+5.1f},{100*b[2]:+5.1f}] {b[3]:4d}"
              + ("   sig" if b[1]>0 or b[2]<0 else ""))
print("reading: predecessor sig -> the reroute PRECEDES the foreign token (a trigger);")
print("         only t sig      -> the reroute coincides with the script (could be reaction).")

# ---- (b)+(c) the occupancy curve on three axes + grapheme weighting --------
def dist_series(r, unit):
    a=audit[r["id"]]; p=r["tip_ai"]; out=[]
    for i in range(max(0,p-24),min(r["n_assistant"],p+25)):
        if unit=="tok": d=i-p
        else:
            f=G if unit=="g" else B
            d=0
            if i<p:  d=-sum(f(r["id"],j) for j in range(i,p))
            elif i>p: d= sum(f(r["id"],j) for j in range(p,i))
        o=occ(r,i)
        if o is not None: out.append((d,o,G(r["id"],i)))
    return out
def curve(recs, tip_of, unit, edges):
    sums=np.zeros(len(edges)-1); wts=np.zeros(len(edges)-1)
    for r in recs:
        p=tip_of(r)
        if p is None or p<1: continue
        rr=dict(r); rr["tip_ai"]=p
        for d,o,g in dist_series(rr,unit):
            k=np.searchsorted(edges,d,side="right")-1
            if 0<=k<len(sums): sums[k]+=o; wts[k]+=1
    return np.where(wts>0,sums/np.maximum(wts,1),np.nan)
rngp=np.random.default_rng(31); tipsx=[r["tip_ai"] for r in T_aud]
def pseudo(c):
    p=int(rngp.choice(tipsx))
    return p if c["n_assistant"]>p+20 else None
AX=[("tok",np.arange(-12.5,13.5,1.0),"model tokens"),
    ("g",  np.arange(-30,33,3.0),   "unicode graphemes"),
    ("b",  np.arange(-60,66,6.0),   "UTF-8 bytes")]
fig,ax=plt.subplots(1,3,figsize=(15,4))
for a_,(unit,edges,lab) in zip(ax,AX):
    ct=curve(T_aud,lambda r:r["tip_ai"],unit,edges)
    cc=curve(C_aud,pseudo,unit,edges)
    x=(edges[:-1]+edges[1:])/2
    a_.plot(x,ct,color="#2563EB",lw=2,label="tipped")
    a_.plot(x,cc,color="#6B7280",lw=2,ls="--",label="control")
    a_.axvline(0,color="black",lw=.8,ls=":")
    a_.set_xlabel(f"distance from onset ({lab})"); a_.set_ylabel("takeover occupancy")
    a_.legend(frameon=False,fontsize=9)
ax[0].set_title("same jump on all three axes = no artifact")
plt.tight_layout(); plt.show()
def wmean(pairs):
    num=sum(o*g for o,g in pairs); den=sum(g for _,g in pairs)
    return num/max(den,1e-9)
for name,tl in (("pre-switch",pre_l),("sustained",sus_l)):
    plain=[]; wpairs=[]
    for rid,idxs in tl:
        rec=next(r for r in T_aud if r["id"]==rid)
        for i in idxs:
            o=occ(rec,i)
            if o is not None: plain.append(o); wpairs.append((o,G(rid,i)))
    if plain:
        print(f"{name:12s} occupancy: token-mean {100*np.mean(plain):5.1f}%   "
              f"grapheme-weighted {100*wmean(wpairs):5.1f}%")

# ---- (d) window-definition stability: token vs content windows -------------
def derive_set(past_fn, min_frac=0.20):
    from collections import Counter
    cnt=Counter()
    for r in T_aud:
        R,A=zl(r); p=r["tip_ai"]; prev=past_fn(r)
        if not prev: continue
        for l in range(L):
            past=set(int(x) for x in R[l,[int(A[j]) for j in prev],:].ravel())
            for e in R[l,int(A[p]),:]:
                if int(e) not in past: cnt[(l,int(e))]+=1
    need=int(np.ceil(min_frac*len(T_aud)))
    return {pe for pe,c in cnt.items() if c>=need}
def tok_win(w):  return lambda r: list(range(max(0,r["tip_ai"]-w),r["tip_ai"]))
def gra_win(g):
    def f(r):
        p=r["tip_ai"]; acc=0; js=[]
        for j in range(p-1,-1,-1):
            js.append(j); acc+=G(r["id"],j)
            if acc>=g: break
        return list(reversed(js))
    return f
print(f"\n{'window definition':24s} {'set size':>8s} {'Jaccard vs canonical':>21s}")
for name,fn in ([(f"tokens W={w}",tok_win(w)) for w in (1,2,4,8,16,32)]
               +[(f"graphemes G={g}",gra_win(g)) for g in (4,8,16,32,64)]):
    S=derive_set(fn)
    j=len(S&CANON)/max(len(S|CANON),1)
    print(f"{name:24s} {len(S):8d} {j:21.3f}")
print("stable Jaccard across token AND content windows = the set is not a tokenization artifact")


### Reading it

* **CAUSAL** (targeted ≪ sham, random ≈ sham): the recurring takeover experts are
  causally necessary for the language switch — the strongest possible closure of
  the phase-5→9b chain (localise → parameterise → intervene).
* **NON-SPECIFIC** (both drop): banning ~equal numbers of experts anywhere hurts
  the behaviour — the takeover set is sufficient-but-not-special. Still
  informative, but no specificity claim.
* **NO EFFECT**: the switch survives without its usual experts — the mode can
  re-route through alternates (fine-grained MoE redundancy). Also a real finding.
* The **consistency gate** (Cell 2) and the **hook verification** (Cell 5) must
  both pass before any generation result is trusted; the sanity line at the end
  checks the ablation did not simply destroy text generation.
